# 🧪 Experiment 09: Decoding vs. Weight Surgery (MLX)

This notebook explores the intersection of **Mechanistic Interpretability** (Weight Surgery) and **Inference Engineering** (Decoding Strategies).

### The Research Question
> "Can advanced decoding strategies rescue a model that is physically collapsing due to weight corruption?"

We will perturb the 4-bit quantized weights of Llama 3 8B and test if techniques like **Top-P**, **Top-K**, and **Contrastive Search** can maintain coherence where Greedy Decoding fails.

### 1. Setup & Model Loading
We use `mlx-lm` to load the 4-bit Llama 3 model. We must store the original weights to restore the model between experiments.

In [1]:
import mlx.core as mx
import mlx.nn as nn
from mlx_lm import load, generate
from mlx_lm.models.cache import KVCache
import numpy as np
import matplotlib.pyplot as plt
import copy

# Load Model & Tokenizer
model_id = "mlx-community/Meta-Llama-3-8B-Instruct-4bit"
model, tokenizer = load(model_id)

# Backup original weights for restoration
original_weight = mx.array(model.lm_head.weight)
original_scales = mx.array(model.lm_head.scales)
original_biases = mx.array(model.lm_head.biases)

print(f"✅ Model Loaded: {model_id}")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

✅ Model Loaded: mlx-community/Meta-Llama-3-8B-Instruct-4bit


### 2. The Weight Surgery Engine

In [2]:
def apply_surgery(shift_amount):
    """
    Manipulates the lm_head weights by a fixed integer shift.
    Thresholds of interest: 1,000,000 (Stable) | 1,000,005 (Collapse)
    """
    model.lm_head.weight = original_weight
    corrupted_weight = (original_weight.astype(mx.int64) - shift_amount).astype(mx.uint32)
    model.lm_head.weight = corrupted_weight
    return hex(corrupted_weight.reshape(-1)[0].item())

### 3. All-in-One Decoding Architecture

In [3]:
def custom_decode(prompt, max_tokens=30, temp=1.0, top_p=1.0, top_k=0, rep_penalty=1.0):
    messages = [{"role": "user", "content": prompt}]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tokens = tokenizer.encode(formatted_prompt)
    input_ids = mx.array(tokens)[None]
    
    generated_tokens = []
    num_layers = len(model.model.layers)
    cache = [KVCache() for _ in range(num_layers)]
    
    # Pre-fill cache
    logits = model(input_ids, cache=cache)
    last_logit = logits[:, -1, :]
    
    for i in range(max_tokens):
        # 1. Repetition Penalty
        if rep_penalty != 1.0:
            for token in set(generated_tokens):
                last_logit[:, token] = mx.where(last_logit[:, token] > 0, last_logit[:, token] / rep_penalty, last_logit[:, token] * rep_penalty)
        
        # 2. Temperature
        if temp != 0 and temp != 1.0:
            last_logit = last_logit / temp
            
        # 3. Top-K
        if top_k > 0:
            top_values = mx.sort(last_logit, axis=-1)[:, -top_k:]
            min_value = top_values[:, 0:1]
            last_logit = mx.where(last_logit < min_value, mx.array(-float('inf')), last_logit)

        # 4. Top-P (Nucleus)
        if top_p < 1.0:
            probs = mx.softmax(last_logit, axis=-1)
            sorted_probs = mx.sort(probs, axis=-1)[:, ::-1]
            cumulative_probs = mx.cumsum(sorted_probs, axis=-1)
            mask = mx.concatenate([mx.zeros((1, 1), dtype=mx.bool_), cumulative_probs[:, :-1] > top_p], axis=-1)
            last_logit = mx.where(mask, mx.array(-float('inf')), last_logit)
            
        # Sample
        if temp == 0:
            next_token = mx.argmax(last_logit, axis=-1)
        else:
            next_token = mx.random.categorical(last_logit)
            
        token_id = next_token.item()
        generated_tokens.append(token_id)
        if token_id == tokenizer.eos_token_id: break
            
        last_logit = model(next_token[None], cache=cache)[:, -1, :]
        
    return tokenizer.decode(generated_tokens).strip()

def contrastive_decode(prompt, max_tokens=30, k=4, alpha=0.6):
    messages = [{"role": "user", "content": prompt}]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tokens = tokenizer.encode(formatted_prompt)
    input_ids = mx.array(tokens)[None]
    
    generated_tokens = []
    past_hidden_states = []
    num_layers = len(model.model.layers)
    cache = [KVCache() for _ in range(num_layers)]
    
    h = model.model(input_ids, cache=cache)
    logits = model.lm_head(h)
    last_h, last_logit = h[:, -1, :], logits[:, -1, :]
    
    for i in range(max_tokens):
        past_hidden_states.append(last_h)
        probs = mx.softmax(last_logit, axis=-1)
        sorted_indices = mx.argsort(probs, axis=-1)[:, ::-1]
        top_k_indices = sorted_indices[:, :k]
        top_k_probs = mx.take_along_axis(probs, top_k_indices, axis=-1)
        
        best_score, best_token = -1e10, None
        candidates, candidate_probs = top_k_indices[0].tolist(), top_k_probs[0].tolist()
        
        for idx, token_id in enumerate(candidates):
            temp_cache = copy.deepcopy(cache)
            h_cand = model.model(mx.array([[token_id]]), cache=temp_cache)[:, -1, :]
            
            all_past = mx.concatenate(past_hidden_states, axis=0)
            h_cand_norm = h_cand / mx.linalg.norm(h_cand, axis=-1, keepdims=True)
            all_past_norm = all_past / mx.linalg.norm(all_past, axis=-1, keepdims=True)
            
            degen_penalty = mx.max(mx.matmul(h_cand_norm, all_past_norm.T)).item()
            score = (1 - alpha) * candidate_probs[idx] - alpha * degen_penalty
            
            if score > best_score:
                best_score, best_token = score, token_id
        
        generated_tokens.append(best_token)
        if best_token == tokenizer.eos_token_id: break
            
        h_final = model.model(mx.array([[best_token]]), cache=cache)
        last_h, last_logit = h_final[:, -1, :], model.lm_head(h_final)[:, -1, :]
        
    return tokenizer.decode(generated_tokens).strip()

### 4. The Grand Redemption Sweep
Iterating through shifts and decoding strategies to find the breaking point and the recovery.

In [4]:
prompt = "What is 2+2?"
shifts = [1_000_000, 1_000_001, 1_000_003, 1_000_005, 1_000_010]

for s in shifts:
    h_val = apply_surgery(s)
    print(f"\n{'='*60}")
    print(f"🔹 WEIGHT SHIFT: {s} | HEX: {h_val}")
    print(f"{'='*60}")
    
    print(f"[Greedy]      : {custom_decode(prompt, temp=0)}")
    print(f"[Temp=0.7]    : {custom_decode(prompt, temp=0.7)}")
    print(f"[Top-K=40]    : {custom_decode(prompt, temp=0.7, top_k=40)}")
    print(f"[Top-P=0.9]    : {custom_decode(prompt, temp=0.7, top_p=0.9)}")
    print(f"[Contrastive] : {contrastive_decode(prompt, k=4, alpha=0.6)}")
    
# Restore
model.lm_head.weight = original_weight
print("\n✅ Weights Restored.")


🔹 WEIGHT SHIFT: 1000000 | HEX: 0x50827458
[Greedy]      : The answer to 2+2 is 4.<|eot_id|>
[Temp=0.7]    : The answer to 2+2 is 4oneyizmetFTWAREdreizmetïantenïantenvorwortantenvorwortantenvorwortantenvorwort
[Top-K=40]    : The answer to 2+2 is 4.<|eot_id|>
[Top-P=0.9]    : !!#E!9I9I$#&9I!$!#!#!$!##%9"9E
[Contrastive] : The answer to 2+2 is 4لكتر<|start_header_id|>assistant<|end_header_id|>

Simple and easy! You're right,_TypeInfodre stoi\views/frontend/>\x

🔹 WEIGHT SHIFT: 1000001 | HEX: 0x50827457
[Greedy]      : The answer to 2+2 is 4.<|eot_id|>
[Temp=0.7]    : The answer is 4! Ansριά…the quickest intervention againstoldur…theernetes…the…the mostuniacid…it to PureComponent65 rounds/sources AS…but vrou lesbihci
[Top-K=40]    : The answer to 2+2 is 4.<|eot_id|>
[Top-P=0.9]    : !!!!!$10$9$&9$&!9I9&!!%&!&!&!&!
[Contrastive] : The answer to 2+2 is 4.<|eot_id|>

🔹 WEIGHT SHIFT: 1000003 | HEX: 0x50827455
[Greedy]      : The answer is 4…andلكتر"... SIMPLEizmet"... nothing…and"... nothing